# EGY-SCOUT V8 — Wave 0 / Sprint 0.1 ($0 GPU Pipeline)
**4-Layer Zero-Referee-Leak Shield on Kaggle Free GPU (T4) & Google Colab (NVIDIA L4)**

This notebook executes the Wave 0 ($0 Investor MVP) pipeline using the `Pluggable Provider` pattern:
1. Auto-detects **Kaggle (T4/P100)** vs **Google Colab (NVIDIA L4)** vs **Local** and tunes batch size & precision (`bf16` on L4, `fp16` on T4).
2. Runs **YOLO + ByteTrack** detection & tracking + **PnLCalib** pitch coordinates (reusing existing calibration if provided).
3. Applies the **4-Layer Zero-Referee-Leak Shield**:
   - **Layer 1:** Detector role prior (`player` / `goalkeeper` / `referee` / `ball`)
   - **Layer 2:** `SigLIP (google/siglip-base-patch16-224)` + `UMAP (3D)` + **3-Class Clustering** (`team_a` / `team_b` / `referee`) + strict `< 0.65` low-confidence rejection
   - **Layer 3:** Touchline geometric rule (`y_m < 1.2m` or `y_m > 66.8m` -> `assistant_referee`)
   - **Layer 4:** **Track-Level Temporal Voting** (100% role stability across each track)
4. Exports **Apache Parquet (Zstd)** (frame-by-frame coordinates), **`summary.json`** (`MATCH_TRACK` metadata + runtime & peak VRAM profiling), and **Before vs. After investor showcase visuals**.


In [ ]:
# Cell 1: Auto-Detect Environment, GPU & Pluggable Provider Config
import os
import sys
import time
import json
import platform
from pathlib import Path


def detect_env():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.path.exists("/kaggle/working"):
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or os.path.exists("/content"):
        return "colab"
    return "local"


ENV_NAME = detect_env()
os.environ.setdefault("EGY_SCOUT_COMPUTE_PROVIDER", ENV_NAME)

if ENV_NAME == "kaggle":
    DEFAULT_DATA_DIR = "/kaggle/working/egy-scout-data"
elif ENV_NAME == "colab":
    DEFAULT_DATA_DIR = "/content/egy-scout-data"
else:
    DEFAULT_DATA_DIR = r"D:\egy-scout-data" if platform.system() == "Windows" else str(Path.home() / "egy-scout-data")

DATA_DIR = Path(os.environ.get("EGY_SCOUT_DATA_DIR", DEFAULT_DATA_DIR))
OUTPUT_DIR = DATA_DIR / "outputs" / "sprint01"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if DEVICE == "cuda" else 0.0

if "L4" in GPU_NAME.upper() or VRAM_GB >= 20.0:
    BATCH_SIZE, PRECISION = 64, "bf16"
elif DEVICE == "cuda" and VRAM_GB >= 12.0:
    BATCH_SIZE, PRECISION = 32, "fp16"
elif DEVICE == "cuda":
    BATCH_SIZE, PRECISION = 16, "fp16"
else:
    BATCH_SIZE, PRECISION = 8, "fp32"

print(f"[ENV] {ENV_NAME.upper()} | Device: {DEVICE} ({GPU_NAME}, {VRAM_GB} GB VRAM)")
print(f"[CFG] Batch Size: {BATCH_SIZE} | Precision: {PRECISION} | Output: {OUTPUT_DIR}")


class StageProfiler:
    def __init__(self):
        self.stages = {}
        self._t0 = None
        self._name = None

    def start(self, name: str):
        if DEVICE == "cuda":
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        self._name = name
        self._t0 = time.perf_counter()

    def stop(self):
        if DEVICE == "cuda":
            torch.cuda.synchronize()
            peak_mb = round(torch.cuda.max_memory_allocated() / (1024**2), 1)
        else:
            peak_mb = 0.0
        elapsed = round(time.perf_counter() - self._t0, 3)
        self.stages[self._name] = {"elapsed_s": elapsed, "peak_vram_mb": peak_mb}
        print(f"[PROFILE] {self._name}: {elapsed:.2f}s | Peak VRAM: {peak_mb:.1f} MB")


PROFILER = StageProfiler()


In [ ]:
# Cell 2: Install Dependencies (Uncomment on a fresh Kaggle / Colab runtime)
# !pip install -q ultralytics supervision umap-learn transformers pyarrow pandas scikit-learn opencv-python-headless


In [ ]:
# Cell 3: Match Inputs & Paths (Edit these paths for your uploaded match)
MATCH_ID = "m2_ahly_enppi"
SPOTLIGHT_FRAME = 1250
FRAME_STRIDE = 4        # Process every Nth frame (1 = full fps, 4 = matches legacy scout5)
MAX_FRAMES = 600        # Set to None to process the full video

VIDEO_PATH = "/kaggle/input/egy-scout-sample/9OcELOXQg2c.mp4" if ENV_NAME == "kaggle" else "/content/9OcELOXQg2c.mp4"
# Optional: Legacy scout5 telemetry.parquet (provides PnLCalib x_m,y_m AND Before-vs-After baseline!)
LEGACY_PARQUET_PATH = "/kaggle/input/egy-scout-sample/telemetry.parquet" if ENV_NAME == "kaggle" else "/content/telemetry.parquet"

# Optional Google Drive mount on Colab:
# if ENV_NAME == "colab":
#     from google.colab import drive
#     drive.mount("/content/drive")

print("Match:", MATCH_ID, "| Video exists:", Path(VIDEO_PATH).is_file(), "| Legacy Parquet exists:", Path(LEGACY_PARQUET_PATH).is_file())


In [ ]:
# Cell 4: Unicode-Safe I/O & 4-Layer Zero-Referee-Leak Shield
import cv2
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from collections import defaultdict
from sklearn.cluster import KMeans

TRACKING_PARQUET_SCHEMA = pa.schema([
    ("match_id", pa.string()),
    ("frame_id", pa.int32()),
    ("time_s", pa.float32()),
    ("track_id", pa.int32()),
    ("role", pa.string()),
    ("team", pa.string()),
    ("role_conf", pa.float32()),
    ("team_conf", pa.float32()),
    ("x1", pa.float32()),
    ("y1", pa.float32()),
    ("x2", pa.float32()),
    ("y2", pa.float32()),
    ("u", pa.float32()),
    ("v", pa.float32()),
    ("x_m", pa.float32()),
    ("y_m", pa.float32()),
    ("on_pitch", pa.bool_()),
    ("jersey_number", pa.int16()),
    ("jersey_conf", pa.float32()),
    ("source_layer", pa.string()),
])


def imwrite_unicode(path, image, params=None):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    ok, enc = cv2.imencode(p.suffix.lower() or ".jpg", image, params or [])
    if ok:
        enc.tofile(str(p))
    return ok


def extract_torso_crop(frame_bgr, bbox, min_size=8):
    h, w = frame_bgr.shape[:2]
    x1, y1, x2, y2 = [float(v) for v in bbox]
    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    tx1 = int(max(0, min(w - 1, round(x1 + 0.20 * bw))))
    tx2 = int(max(tx1 + 1, min(w, round(x1 + 0.80 * bw))))
    ty1 = int(max(0, min(h - 1, round(y1 + 0.15 * bh))))
    ty2 = int(max(ty1 + 1, min(h, round(y1 + 0.58 * bh))))
    crop = frame_bgr[ty1:ty2, tx1:tx2]
    return crop if (crop.shape[0] >= min_size and crop.shape[1] >= min_size) else None


def compute_cluster_confidences(points, centroids):
    dists = np.linalg.norm(points[:, None, :] - centroids[None, :, :], axis=2)
    order = np.argsort(dists, axis=1)
    labels = order[:, 0]
    pair_dists = [
        float(np.linalg.norm(centroids[i] - centroids[j]))
        for i in range(len(centroids))
        for j in range(i + 1, len(centroids))
    ]
    scale = max(float(np.median(pair_dists)) * 0.45, 1e-4) if pair_dists else 1.0
    if len(centroids) >= 2:
        d1 = dists[np.arange(len(points)), order[:, 0]]
        d2 = dists[np.arange(len(points)), order[:, 1]]
        margin = np.clip((d2 - d1) / np.clip(d2, 1e-6, None), 0.0, 1.0)
    else:
        margin = np.ones(len(points), dtype=np.float32)
    logits = -(dists / scale)
    logits -= logits.max(axis=1, keepdims=True)
    probs = np.exp(logits)
    probs /= np.clip(probs.sum(axis=1, keepdims=True), 1e-8, None)
    p_top = probs[np.arange(len(points)), labels]
    return labels.astype(int), np.clip(0.5 * p_top + 0.5 * margin, 0.0, 1.0).astype(np.float32)


class SigLIPZeroLeakShield:
    def __init__(
        self,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        precision=PRECISION,
        min_team_conf=0.65,
        touchline_margin_m=1.2,
        pitch_w=68.0,
        pitch_l=105.0,
    ):
        self.device = device
        self.batch_size = batch_size
        self.precision = precision
        self.min_team_conf = min_team_conf
        self.touchline_margin_m = touchline_margin_m
        self.pitch_w = pitch_w
        self.pitch_l = pitch_l

    def extract_siglip_embeddings(self, crops):
        from PIL import Image
        from transformers import AutoProcessor, SiglipVisionModel

        model_id = "google/siglip-base-patch16-224"
        processor = AutoProcessor.from_pretrained(model_id)
        model = SiglipVisionModel.from_pretrained(model_id).to(self.device)
        model.eval()

        pil_crops = [Image.fromarray(cv2.cvtColor(c, cv2.COLOR_BGR2RGB)) for c in crops]
        amp_dtype = (
            torch.bfloat16
            if self.precision == "bf16"
            else (torch.float16 if self.precision == "fp16" else torch.float32)
        )
        all_embs = []
        with torch.no_grad():
            for i in range(0, len(pil_crops), self.batch_size):
                batch = pil_crops[i : i + self.batch_size]
                inputs = processor(images=batch, return_tensors="pt").to(self.device)
                if self.device == "cuda" and self.precision in ("bf16", "fp16"):
                    with torch.amp.autocast("cuda", dtype=amp_dtype):
                        out = model(**inputs)
                else:
                    out = model(**inputs)
                emb = torch.mean(out.last_hidden_state, dim=1).float().cpu().numpy()
                all_embs.append(emb)
        return np.concatenate(all_embs, axis=0)

    def run(self, df: pd.DataFrame, crops: list):
        import umap

        out = df.copy().reset_index(drop=True)
        valid_idx = [i for i, c in enumerate(crops) if c is not None]
        valid_crops = [crops[i] for i in valid_idx]
        if len(valid_crops) < 3:
            raise ValueError(f"Need at least 3 valid torso crops for 3-class clustering, got {len(valid_crops)}.")

        # Layer 2: SigLIP + UMAP + 3-Class KMeans
        PROFILER.start("layer2_siglip_umap_3class")
        embs = self.extract_siglip_embeddings(valid_crops)
        n_neighbors = min(15, max(3, len(embs) - 1))
        reducer = umap.UMAP(n_components=3, n_neighbors=n_neighbors, random_state=42)
        proj = reducer.fit_transform(embs)
        km = KMeans(n_clusters=3, n_init=10, random_state=42)
        raw_labels = km.fit_predict(proj)
        cids, confs = compute_cluster_confidences(proj, km.cluster_centers_)

        counts = sorted([(int((raw_labels == cid).sum()), cid) for cid in range(3)], reverse=True)
        team_cids = sorted([counts[0][1], counts[1][1]])
        cluster_map = {
            team_cids[0]: ("player", "team_a"),
            team_cids[1]: ("player", "team_b"),
            counts[2][1]: ("referee", "none"),
        }
        l2_map = {}
        for idx, cid, conf in zip(valid_idx, cids, confs):
            r, t = cluster_map[int(cid)]
            if r == "player" and float(conf) < self.min_team_conf:
                l2_map[idx] = ("referee", "none", 1.0 - float(conf), "layer2_low_conf_reject")
            elif r == "referee":
                l2_map[idx] = ("referee", "none", float(conf), "layer2_3class_outlier")
            else:
                l2_map[idx] = (r, t, float(conf), "layer2_siglip_3class")
        PROFILER.stop()

        # Layer 3: Touchline Geometric Rule + Merge with Layer 1/2
        PROFILER.start("layer3_4_touchline_and_temporal_vote")
        touch_flags = []
        roles, teams, rconfs, tconfs, on_pitches, layers = [], [], [], [], [], []
        for i, row in out.iterrows():
            xm, ym = row.get("x_m", np.nan), row.get("y_m", np.nan)
            in_x = pd.notna(xm) and (-5.0 <= float(xm) <= self.pitch_l + 5.0)
            on_tl = pd.notna(ym) and (
                (-5.0 <= float(ym) < self.touchline_margin_m)
                or ((self.pitch_w - self.touchline_margin_m) < float(ym) <= self.pitch_w + 5.0)
            )
            if in_x and on_tl:
                roles.append("assistant_referee")
                teams.append("none")
                rconfs.append(0.96)
                tconfs.append(0.0)
                on_pitches.append(False)
                layers.append("layer3_touchline_rule")
                touch_flags.append(True)
                continue
            touch_flags.append(False)
            if i in l2_map:
                r2, t2, c2, s2 = l2_map[i]
                roles.append(r2)
                teams.append(t2)
                rconfs.append(c2)
                tconfs.append(c2 if t2 != "none" else 0.0)
                on_pitches.append(bool(row.get("on_pitch", True)))
                layers.append(s2)
            else:
                roles.append("player")
                teams.append("none")
                rconfs.append(0.5)
                tconfs.append(0.0)
                on_pitches.append(bool(row.get("on_pitch", True)))
                layers.append("layer1_passthrough")

        out["role"] = roles
        out["team"] = teams
        out["role_conf"] = np.asarray(rconfs, dtype=np.float32)
        out["team_conf"] = np.asarray(tconfs, dtype=np.float32)
        out["on_pitch"] = on_pitches
        out["source_layer"] = layers
        out["_tl"] = touch_flags

        # Layer 4: Track-Level Temporal Voting + Spatial Penalty-Box Goalkeeper Separation
        team_x_med = {}
        for tname in ("team_a", "team_b"):
            sub_t = out[(out["role"] == "player") & (out["team"] == tname) & out["x_m"].notna()]
            if not sub_t.empty:
                team_x_med[tname] = float(sub_t["x_m"].median())

        weights = {
            "layer3_touchline_rule": 2.5,
            "layer2_3class_outlier": 1.5,
            "layer2_low_conf_reject": 1.3,
            "layer2_siglip_3class": 1.2,
            "layer1_passthrough": 1.0,
        }
        for tid, idxs in out.groupby("track_id").groups.items():
            if int(tid) < 0:
                continue
            grp = out.loc[idxs]
            if grp["y_m"].notna().any() and float(grp.loc[grp["y_m"].notna(), "_tl"].mean()) >= 0.40:
                out.loc[idxs, "role"] = "assistant_referee"
                out.loc[idxs, "team"] = "none"
                out.loc[idxs, "on_pitch"] = False
                out.loc[idxs, "role_conf"] = 0.96
                out.loc[idxs, "team_conf"] = 0.0
                out.loc[idxs, "source_layer"] = "layer4_vote_touchline"
                continue
            r_scores, t_scores = defaultdict(float), defaultdict(float)
            for _, r in grp.iterrows():
                w = weights.get(str(r["source_layer"]), 1.0)
                r_scores[str(r["role"])] += w * max(0.1, float(r["role_conf"]))
                if str(r["team"]) in ("team_a", "team_b"):
                    t_scores[str(r["team"])] += w * max(0.1, float(r["team_conf"]))
            best_r = max(r_scores.items(), key=lambda kv: kv[1])[0]

            # Spatial penalty-box GK separation (prevents GK from being labeled referee)
            cal_gk = grp[grp["x_m"].notna() & grp["y_m"].notna()]
            gk_end = None
            if not cal_gk.empty:
                xs, ys = cal_gk["x_m"].to_numpy(dtype=np.float32), cal_gk["y_m"].to_numpy(dtype=np.float32)
                if float(xs.max() - xs.min()) <= 28.0:
                    in_y = (ys >= 13.84) & (ys <= 54.16)
                    if float(((xs >= -1.0) & (xs <= 16.5) & in_y).mean()) >= 0.55:
                        gk_end = "left"
                    elif float(((xs >= self.pitch_l - 16.5) & (xs <= self.pitch_l + 1.0) & in_y).mean()) >= 0.55:
                        gk_end = "right"
            if gk_end is not None and (best_r in ("referee", "goalkeeper") or float(grp["team_conf"].mean()) < self.min_team_conf):
                goal_x = 0.0 if gk_end == "left" else self.pitch_l
                gk_team = min(team_x_med.items(), key=lambda kv: abs(kv[1] - goal_x))[0] if team_x_med else ("team_a" if gk_end == "left" else "team_b")
                out.loc[idxs, "role"] = "goalkeeper"
                out.loc[idxs, "team"] = gk_team
                out.loc[idxs, "on_pitch"] = True
                out.loc[idxs, "role_conf"] = 0.92
                out.loc[idxs, "team_conf"] = 0.90
                out.loc[idxs, "source_layer"] = "layer4_spatial_gk_box"
                continue

            best_t = (
                "none"
                if best_r in ("referee", "assistant_referee", "staff")
                else (max(t_scores.items(), key=lambda kv: kv[1])[0] if t_scores else "none")
            )
            out.loc[idxs, "role"] = best_r
            out.loc[idxs, "team"] = best_t
            out.loc[idxs, "role_conf"] = round(r_scores[best_r] / (sum(r_scores.values()) or 1.0), 4)
            out.loc[idxs, "team_conf"] = (
                round(t_scores[best_t] / (sum(t_scores.values()) or 1.0), 4) if best_t != "none" else 0.0
            )
            out.loc[idxs, "source_layer"] = "layer4_temporal_vote"

        PROFILER.stop()
        return out.drop(columns=["_tl"])


In [ ]:
# Cell 5: Load Detections (Reuse Legacy Telemetry+PnLCalib OR Run Fresh YOLO+ByteTrack)
PROFILER.start("layer1_detection_and_crops")
cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), f"Cannot open video: {VIDEO_PATH}"
fps = float(cap.get(cv2.CAP_PROP_FPS)) or 25.0

legacy_df = None
if Path(LEGACY_PARQUET_PATH).is_file():
    raw = pd.read_parquet(LEGACY_PARQUET_PATH)
    team_map = {0: "team_a", 1: "team_b"}
    legacy_df = pd.DataFrame({
        "match_id": MATCH_ID,
        "frame_id": raw["frame_id"].astype("int32"),
        "time_s": raw.get("time_s", raw["frame_id"] / fps).astype("float32"),
        "track_id": raw["track_id"].astype("int32"),
        "role": raw.get("role", "player").astype(str),
        "team": raw["team_id"].map(lambda x: team_map.get(int(x), "none") if pd.notna(x) else "none") if "team_id" in raw.columns else "none",
        "role_conf": raw.get("conf", 0.8).astype("float32"),
        "team_conf": raw.get("team_conf", 0.0).astype("float32"),
        "x1": raw["x1"].astype("float32"), "y1": raw["y1"].astype("float32"),
        "x2": raw["x2"].astype("float32"), "y2": raw["y2"].astype("float32"),
        "u": raw.get("u", (raw["x1"] + raw["x2"]) / 2.0).astype("float32"),
        "v": raw.get("v", raw["y2"]).astype("float32"),
        "x_m": raw.get("x_m", np.nan).astype("float32"),
        "y_m": raw.get("y_m", np.nan).astype("float32"),
        "on_pitch": raw.get("on_pitch", True).astype(bool),
        "jersey_number": np.int16(-1), "jersey_conf": np.float32(0.0),
        "source_layer": "legacy_scout5",
    })
    det_df = legacy_df[legacy_df["role"] != "ball"].copy().reset_index(drop=True)
    print(f"[INFO] Loaded {len(det_df)} calibrated detections from legacy PnLCalib Parquet.")
else:
    from ultralytics import YOLO
    yolo = YOLO("yolo11m.pt")
    rows = []
    fid = 0
    processed = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if fid % FRAME_STRIDE == 0:
            res = yolo.track(frame, persist=True, classes=[0], conf=0.35, tracker="bytetrack.yaml", verbose=False)[0]
            if res.boxes is not None and res.boxes.id is not None:
                for box, tid, conf in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.id.int().cpu().numpy(), res.boxes.conf.cpu().numpy()):
                    x1, y1, x2, y2 = [float(v) for v in box]
                    rows.append({
                        "match_id": MATCH_ID, "frame_id": fid, "time_s": fid / fps,
                        "track_id": int(tid), "role": "player", "team": "none",
                        "role_conf": float(conf), "team_conf": 0.0,
                        "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                        "u": (x1 + x2) / 2.0, "v": y2, "x_m": np.nan, "y_m": np.nan,
                        "on_pitch": True, "jersey_number": -1, "jersey_conf": 0.0,
                        "source_layer": "yolo11_bytetrack",
                    })
            processed += 1
            if MAX_FRAMES and processed >= MAX_FRAMES:
                break
        fid += 1
    det_df = pd.DataFrame(rows)

crops = [None] * len(det_df)
for fid, grp in det_df.groupby("frame_id", sort=True):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(fid))
    ok, frame_bgr = cap.read()
    if not ok:
        continue
    for idx, r in grp.iterrows():
        crops[idx] = extract_torso_crop(frame_bgr, [r["x1"], r["y1"], r["x2"], r["y2"]])
cap.release()
PROFILER.stop()
print(f"[INFO] Extracted {sum(c is not None for c in crops)}/{len(crops)} torso crops.")


In [ ]:
# Cell 6: Run 4-Layer Zero-Referee-Leak Shield & Export Parquet + Summary JSON
shield = SigLIPZeroLeakShield(device=DEVICE, batch_size=BATCH_SIZE, precision=PRECISION)
v8_df = shield.run(det_df, crops)

# Export Apache Parquet (Zstd) — frame-by-frame coordinates ONLY in Parquet
parquet_out = OUTPUT_DIR / f"{MATCH_ID}_tracking_v8.parquet"
ordered_cols = [f.name for f in TRACKING_PARQUET_SCHEMA]
table = pa.Table.from_pandas(v8_df[ordered_cols], schema=TRACKING_PARQUET_SCHEMA, preserve_index=False)
pq.write_table(table, str(parquet_out), compression="zstd")

# Build PostgreSQL MATCH_TRACK metadata summary (metadata ONLY, no frame coordinates)
match_tracks_meta = []
for tid, grp in v8_df.groupby("track_id", sort=True):
    match_tracks_meta.append({
        "match_id": MATCH_ID,
        "track_id": int(tid),
        "role": str(grp["role"].mode().iloc[0]),
        "team": str(grp["team"].mode().iloc[0]),
        "role_conf": round(float(grp["role_conf"].mean()), 4),
        "team_conf": round(float(grp["team_conf"].mean()), 4),
        "first_frame": int(grp["frame_id"].min()),
        "last_frame": int(grp["frame_id"].max()),
        "frame_count": int(len(grp)),
        "parquet_uri": str(parquet_out),
    })

summary = {
    "match_id": MATCH_ID,
    "compute_provider": ENV_NAME,
    "gpu_name": GPU_NAME,
    "vram_total_gb": VRAM_GB,
    "batch_size": BATCH_SIZE,
    "precision": PRECISION,
    "total_rows": int(len(v8_df)),
    "unique_tracks": int(v8_df["track_id"].nunique()),
    "role_distribution": v8_df["role"].value_counts().to_dict(),
    "team_distribution": v8_df["team"].value_counts().to_dict(),
    "profiling": PROFILER.stages,
    "match_tracks": match_tracks_meta,
}
summary_out = OUTPUT_DIR / f"{MATCH_ID}_summary.json"
summary_out.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("[OK] Parquet:", parquet_out, f"({parquet_out.stat().st_size / 1024:.1f} KB)")
print("[OK] Summary:", summary_out)
print("Roles:", summary["role_distribution"])


In [ ]:
# Cell 7: Investor Showcase — Before vs. After Side-by-Side Image & MP4 Clip
COLOR_MAP = {
    ("player", "team_a"): (246, 130, 59),     # Blue (BGR)
    ("player", "team_b"): (68, 68, 239),      # Red
    ("goalkeeper", "team_a"): (212, 182, 6),  # Cyan
    ("goalkeeper", "team_b"): (212, 182, 6),
    ("referee", "none"): (21, 204, 250),      # Bright Yellow
    ("assistant_referee", "none"): (22, 115, 249),  # Orange
}


def draw_boxes(frame_bgr, sub_df, title):
    vis = frame_bgr.copy()
    cv2.rectangle(vis, (0, 0), (vis.shape[1], 42), (15, 23, 42), -1)
    cv2.putText(vis, title, (16, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    for _, r in sub_df.iterrows():
        x1, y1, x2, y2 = int(r["x1"]), int(r["y1"]), int(r["x2"]), int(r["y2"])
        color = COLOR_MAP.get((str(r["role"]), str(r["team"])), (180, 180, 180))
        thick = 3 if str(r["role"]) in ("referee", "assistant_referee") else 2
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, thick)
        lbl = f"T{int(r['track_id'])} {r['role'][:3].upper()}/{r['team']}"
        cv2.putText(vis, lbl, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.48, color, 2, cv2.LINE_AA)
    return vis


cap = cv2.VideoCapture(str(VIDEO_PATH))
avail_fids = sorted(v8_df["frame_id"].unique().tolist())
target_fid = SPOTLIGHT_FRAME if SPOTLIGHT_FRAME in avail_fids else (avail_fids[0] if avail_fids else SPOTLIGHT_FRAME)
cap.set(cv2.CAP_PROP_POS_FRAMES, int(target_fid))
ok, frame_bgr = cap.read()
if ok and frame_bgr is not None:
    right = draw_boxes(frame_bgr, v8_df[v8_df["frame_id"] == target_fid], f"AFTER: EGY-SCOUT V8 4-Layer Shield (Frame {target_fid})")
    if legacy_df is not None:
        left = draw_boxes(frame_bgr, legacy_df[legacy_df["frame_id"] == target_fid], f"BEFORE: Legacy 2-Team KMeans (Frame {target_fid})")
        comparison = np.hstack([left, right])
    else:
        comparison = right
    comp_img = OUTPUT_DIR / f"{MATCH_ID}_frame{target_fid}_before_after.jpg"
    imwrite_unicode(comp_img, comparison, [cv2.IMWRITE_JPEG_QUALITY, 94])
    print("[OK] Investor comparison image saved:", comp_img)

clip_frames = sorted(v8_df["frame_id"].unique().tolist())[:60]
if clip_frames:
    w_out = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) * (2 if legacy_df is not None else 1)
    h_out = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    clip_path = OUTPUT_DIR / f"{MATCH_ID}_investor_before_after.mp4"
    writer = cv2.VideoWriter(str(clip_path), cv2.VideoWriter_fourcc(*"mp4v"), 12.0, (w_out, h_out))
    for fid in clip_frames:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(fid))
        ok, fr = cap.read()
        if not ok:
            continue
        r_vis = draw_boxes(fr, v8_df[v8_df["frame_id"] == fid], f"AFTER V8 Shield (F{fid})")
        if legacy_df is not None:
            l_vis = draw_boxes(fr, legacy_df[legacy_df["frame_id"] == fid], f"BEFORE Legacy (F{fid})")
            writer.write(np.hstack([l_vis, r_vis]))
        else:
            writer.write(r_vis)
    writer.release()
    print("[OK] Investor comparison MP4 saved:", clip_path)
cap.release()
